# FloodShift 06 — Counterfactual SAR vs Optical Attribution

This notebook uses the **accepted 04H-CAL model** only.

It does **no training**, does **not** load the rejected 05B weights, and does **not** open the frozen 58-scene geographic test.

The scientific question is:

> For each prediction, how dependent is the model on SAR evidence and how dependent is it on optical evidence?

We evaluate three frozen inference paths:

\[
P_{\mathrm{full}}
\]

\[
P_{-\mathrm{SAR}}
\]

\[
P_{-\mathrm{OPT}}
\]

where:

- \(P_{\mathrm{full}}\): normal SAR + optical + \(P(\mathrm{clear})\)
- \(P_{-\mathrm{SAR}}\): SAR tensor is zeroed; optical and its reliability remain available
- \(P_{-\mathrm{OPT}}\): optical tensor is zeroed and \(P(\mathrm{clear})=0\)

The counterfactual change is measured with **normalized Jensen–Shannon divergence**:

\[
A_{\mathrm{SAR}}
=
\frac{
JS(P_{\mathrm{full}},P_{-\mathrm{SAR}})
}{
\log 2
}
\]

\[
A_{\mathrm{OPT}}
=
\frac{
JS(P_{\mathrm{full}},P_{-\mathrm{OPT}})
}{
\log 2
}
\]

Both scores lie in \([0,1]\).

We also compute:

- prediction flip rate when each modality is removed,
- signed support for the original full-model prediction,
- SAR vs optical contribution share,
- total counterfactual fragility,
- class-specific attribution,
- scene-level attribution summaries,
- relation between \(P(\mathrm{clear})\) and optical/SAR contribution,
- whether counterfactual fragility helps identify prediction errors.

## Important interpretation

These are **model-dependence / counterfactual sensitivity** measurements.

They are **not** a claim of physical causality in the Earth system.

## Frozen lineage

Primary segmenter:

```text
FloodShift_04H_best_mIoU.pt
SHA256:
ef3341a7579f1316c80290dcd2a8f972c0adbf7e03e7a8c6bc239f30c4b47e53
```

Fixed water-type calibration:

```text
type_bias = -0.15
```

Frozen confidence temperature from Notebook 05:

```text
T = 0.989592432975769
```

The 68-scene validation split is deterministically divided exactly as in Notebook 05:

- 34 scenes: attribution calibration / threshold freeze
- 34 scenes: counterfactual evaluation

The frozen Pakistan + Spain test remains untouched.


In [ ]:
# ============================================================
# CELL: P14-C00 — KAGGLE-COMPAT TERRATORCH INSTALL
# ============================================================
# Proven compatible path from 04E.
# IMPORTANT: on a fresh Kaggle image, pip may change NumPy to 1.26.4.
# If the next cell reports a NumPy/Pandas ABI error, restart the Python
# kernel ONCE, do NOT rerun this cell, then continue from P10-C01.

import sys, subprocess

print("Installing TerraTorch 1.1.1 (04E/04F-proven compatibility path)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir", "terratorch==1.1.1"
])
print("P14-C00 COMPLETE ✅")
print("If NumPy changed, restart the Python kernel once before P10-C01.")

In [ ]:
# ============================================================
# CELL: P14-C01 — ENVIRONMENT + REPRODUCIBILITY
# ============================================================
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, json, time, math, random, hashlib, zipfile
from pathlib import Path

import numpy as np
import scipy
from scipy.stats import spearmanr
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
from rasterio.windows import Window

from sklearn.metrics import roc_auc_score, average_precision_score

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_06_Counterfactual_Attribution")
METRIC_ROOT = OUTPUT_ROOT / "metrics"
EVIDENCE_ROOT = OUTPUT_ROOT / "evidence"

for p in [OUTPUT_ROOT, METRIC_ROOT, EVIDENCE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 112)
print("P14-C01 — NOTEBOOK 06 ENVIRONMENT")
print("=" * 112)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("SciPy       :", scipy.__version__)
print("PyTorch     :", torch.__version__)
print("TerraTorch  :", getattr(terratorch, "__version__", "unknown"))
print("Device      :", DEVICE)
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("AMP         :", AMP_ENABLED)
print("Output root :", OUTPUT_ROOT)


In [ ]:
# ============================================================
# CELL: P14-C02 — LOCATE + VERIFY ACCEPTED 04H-CAL LINEAGE
# ============================================================
_file_cache = {}

def locate_by_saved_path(saved_path_or_name, required=True):
    filename = Path(str(saved_path_or_name)).name
    key = str(saved_path_or_name)

    if key in _file_cache:
        return _file_cache[key]

    matches = []
    if INPUT_ROOT.exists():
        matches.extend(INPUT_ROOT.rglob(filename))

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not locate '{filename}' under /kaggle/input."
            )
        return None

    selected = sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]
    _file_cache[key] = selected
    return selected

FINAL_MANIFEST_PATH = locate_by_saved_path("floodshift_preprocessing_manifest_v1.csv")
GLOBAL_LOCK_PATH = locate_by_saved_path("floodshift_preprocessing_GLOBAL_LOCK_v1.json")
OPTICAL_NORM_PATH = locate_by_saved_path("optical_10band_train_normalization_exact.csv")
SAR_SCENE_STATS_PATH = locate_by_saved_path("train_raw_sar_scene_statistics.csv")

CHECKPOINT_04H = locate_by_saved_path("FloodShift_04H_best_mIoU.pt")
CAL_CONFIG_PATH = locate_by_saved_path("FloodShift_04H_CAL_config.pt")

# Notebook-05 uncertainty config is optional; if attached, verify the frozen T.
UNCERTAINTY_05_PATH = locate_by_saved_path(
    "FloodShift_05_uncertainty_config.pt",
    required=False,
)

EXPECTED_04H_SHA256 = "ef3341a7579f1316c80290dcd2a8f972c0adbf7e03e7a8c6bc239f30c4b47e53"
EXPECTED_04E_SHA256 = "6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822"
EXPECTED_TYPE_BIAS = -0.15
FROZEN_TEMPERATURE = 0.989592432975769

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

SHA_04H = sha256_file(CHECKPOINT_04H)
assert SHA_04H == EXPECTED_04H_SHA256

cal_cfg = torch.load(CAL_CONFIG_PATH, map_location="cpu", weights_only=False)
assert cal_cfg["source_04h_checkpoint_sha256"] == EXPECTED_04H_SHA256
assert cal_cfg["source_04e_sha256"] == EXPECTED_04E_SHA256
assert abs(float(cal_cfg["type_bias"]) - EXPECTED_TYPE_BIAS) < 1e-12
assert cal_cfg["decision"] == "04H_CAL_SINGLE_MODEL_PREFERRED"

TYPE_BIAS = float(cal_cfg["type_bias"])

if UNCERTAINTY_05_PATH is not None:
    unc05 = torch.load(
        UNCERTAINTY_05_PATH,
        map_location="cpu",
        weights_only=False,
    )
    assert unc05["source_04h_sha256"] == EXPECTED_04H_SHA256
    assert abs(float(unc05["type_bias"]) - EXPECTED_TYPE_BIAS) < 1e-12
    assert abs(float(unc05["temperature"]) - FROZEN_TEMPERATURE) < 1e-9
    print("Notebook-05 uncertainty config verified:", UNCERTAINTY_05_PATH)
else:
    print("Notebook-05 uncertainty config not attached.")
    print("Using its previously frozen temperature constant:", FROZEN_TEMPERATURE)

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)

assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 112)
print("P14-C02 — ACCEPTED MODEL VERIFIED")
print("=" * 112)
print("04H checkpoint :", CHECKPOINT_04H)
print("04H SHA256     :", SHA_04H)
print("Type bias      :", TYPE_BIAS)
print("Temperature    :", FROZEN_TEMPERATURE)
print("Preprocess     :", preprocessing_lock["status"])
print("05B weights    : NOT LOADED")
print("P14-C02 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P14-C03 — MANIFEST + FROZEN TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# Original S2 source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = (
        np.isfinite(mu_all)
        & np.isfinite(sd_all)
        & (n_all > 0)
    )

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db",
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db",
)

SAR_MEAN = np.array(
    [VV_MEAN, VH_MEAN],
    dtype=np.float32,
)

SAR_STD = np.array(
    [VV_STD, VH_STD],
    dtype=np.float32,
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 112)
print("P14-C03 — DATA CONFIG")
print("=" * 112)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : frozen continuous P(clear)")
print("Primary SAR input  : raw VV/VH")

In [ ]:
# ============================================================
# CELL: P14-C04 — TARGET + DATASET
#
# IMPORTANT:
# This intentionally preserves the CURRENT project's target mapping.
# It does NOT silently change label semantics mid-experiment.
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(
            row["optical_reliability_path"]
        ),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8,
    )

    valid = flood != -1

    target[valid] = 0

    # Current frozen project semantics:
    # any valid JRC permanent-water pixel -> class 1
    target[
        valid & (jrc == 1)
    ] = 1

    # flood-positive but not JRC permanent water -> class 2
    target[
        valid
        & (jrc != 1)
        & (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False,
    )

    valid = np.isfinite(sar).all(axis=0)

    out = np.zeros_like(
        sar,
        dtype=np.float32,
    )

    for c in range(2):
        m = valid & np.isfinite(sar[c])

        out[c, m] = (
            sar[c, m] - SAR_MEAN[c]
        ) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False,
    )

    valid = np.all(
        np.isfinite(optical_13)
        & (optical_13 != 0),
        axis=0,
    )

    reflectance = optical_13 / 10000.0

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    out = (
        optical_10
        - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(
        np.float32,
        copy=False,
    )

    # Frozen Notebook-03 convention.
    out[:, ~valid] = 0.0

    return out


def augment_all(
    sar,
    optical,
    reliability,
    target,
):
    # D4-aligned augmentation only.
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(
            reliability,
            axis=1,
        )
        target = np.flip(
            target,
            axis=1,
        )

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(
            reliability,
            axis=0,
        )
        target = np.flip(
            target,
            axis=0,
        )

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(
            sar,
            k=k,
            axes=(1, 2),
        )

        optical = np.rot90(
            optical,
            k=k,
            axes=(1, 2),
        )

        reliability = np.rot90(
            reliability,
            k=k,
            axes=(0, 1),
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1),
        )

    return (
        sar.copy(),
        optical.copy(),
        reliability.copy(),
        target.copy(),
    )


class FixedEvalDataset(Dataset):
    def __init__(
        self,
        dataframe,
        train_mode=False,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            * TILES_PER_SCENE
        )

    def __getitem__(self, index):
        scene_index = (
            index // TILES_PER_SCENE
        )

        tile_index = (
            index % TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE,
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window,
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window,
            )

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r",
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE,
            ],
            dtype=np.float32,
        )

        sar = normalize_sar(sar)

        optical = normalize_optical(
            optical_13
        )

        target = build_three_class_target(
            flood,
            jrc,
        )

        reliability = np.clip(
            reliability,
            0.0,
            1.0,
        )

        if self.train_mode:
            (
                sar,
                optical,
                reliability,
                target,
            ) = augment_all(
                sar,
                optical,
                reliability,
                target,
            )

        return {
            "sar": torch.from_numpy(
                sar
            ),
            "optical": torch.from_numpy(
                optical
            ),
            "reliability": torch.from_numpy(
                reliability[
                    None,
                    ...,
                ]
            ),
            "target": torch.from_numpy(
                target.astype(
                    np.int64
                )
            ),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }

In [ ]:
# ============================================================
# CELL: P14-C05 — SAME DETERMINISTIC VAL PARTITION AS NOTEBOOK 05
# ============================================================
VAL_BATCH_SIZE = 2
NUM_WORKERS = 2

def stable_scene_hash(scene_id):
    token = ("FloodShift05|" + str(scene_id)).encode("utf-8")
    return hashlib.sha256(token).hexdigest()

val_partition = val_df[["scene_id"]].copy()
val_partition["partition_hash"] = val_partition["scene_id"].map(stable_scene_hash)
val_partition = val_partition.sort_values("partition_hash").reset_index(drop=True)

calibration_ids = set(val_partition.iloc[:34]["scene_id"].astype(str))
attribution_ids = set(val_partition.iloc[34:]["scene_id"].astype(str))

assert len(calibration_ids) == 34
assert len(attribution_ids) == 34
assert calibration_ids.isdisjoint(attribution_ids)

val_cal_df = val_df[val_df["scene_id"].astype(str).isin(calibration_ids)].copy().reset_index(drop=True)
val_attr_df = val_df[val_df["scene_id"].astype(str).isin(attribution_ids)].copy().reset_index(drop=True)

assert len(val_cal_df) == 34
assert len(val_attr_df) == 34

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

common_loader = dict(
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    persistent_workers=(NUM_WORKERS > 0),
)

val_cal_loader = DataLoader(
    FixedEvalDataset(val_cal_df, train_mode=False),
    **common_loader,
)

val_attr_loader = DataLoader(
    FixedEvalDataset(val_attr_df, train_mode=False),
    **common_loader,
)

print("=" * 112)
print("P14-C05 — ATTRIBUTION PARTITION")
print("=" * 112)
print("Calibration scenes :", len(val_cal_df))
print("Attribution scenes :", len(val_attr_df))
print("Frozen test scenes :", len(test_df), "(LOCKED; no Dataset/DataLoader constructed)")
print("Calibration batches:", len(val_cal_loader))
print("Attribution batches:", len(val_attr_loader))
assert len(test_df) == 58
print("P14-C05 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P14-C06 — FINAL TERRAMIND-RIFT ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# TerraMind pretrained input conventions
# Official Sen1Floods11 config statistics.
# ------------------------------------------------------------

TM_S2_ALL_MEAN = np.array(
    [
        2357.089,
        2137.385,
        2018.788,
        2082.986,
        2295.651,
        2854.537,
        3122.849,
        3040.560,
        3306.481,
        1473.847,
        506.070,
        2472.825,
        1838.929,
    ],
    dtype=np.float32,
)

TM_S2_ALL_STD = np.array(
    [
        1624.683,
        1675.806,
        1557.708,
        1833.702,
        1823.738,
        1733.977,
        1732.131,
        1679.732,
        1727.260,
        1024.687,
        442.165,
        1331.411,
        1160.419,
    ],
    dtype=np.float32,
)

TM_S2_SUBSET_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

TM_S2_MEAN = (
    TM_S2_ALL_MEAN[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S2_STD = (
    TM_S2_ALL_STD[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S1_MEAN = np.array(
    [-12.599, -20.293],
    dtype=np.float32,
)

TM_S1_STD = np.array(
    [5.195, 5.890],
    dtype=np.float32,
)

TM_OPTICAL_BAND_NAMES = [
    "BLUE",
    "GREEN",
    "RED",
    "RED_EDGE_1",
    "RED_EDGE_2",
    "RED_EDGE_3",
    "NIR_BROAD",
    "NIR_NARROW",
    "SWIR_1",
    "SWIR_2",
]

TM_LAYER_INDICES = [
    2,
    5,
    8,
    11,
]

TM_EMBED_DIM = 768
PYRAMID_CHANNELS = [
    64,
    128,
    256,
    512,
]

PYRAMID_SIZES = [
    128,
    64,
    32,
    16,
]


def make_group_norm(channels):
    for groups in [
        32,
        16,
        8,
        4,
        2,
        1,
    ]:
        if (
            channels
            % groups
            == 0
        ):
            return nn.GroupNorm(
                groups,
                channels,
            )

    raise RuntimeError(
        "No GroupNorm divisor."
    )


class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
    ):
        super().__init__()

        padding = (
            kernel_size // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(
        self,
        channels,
        reduction=8,
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            8,
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(self, x):
        w = F.adaptive_avg_pool2d(
            x,
            1,
        )

        w = F.gelu(
            self.fc1(w)
        )

        w = torch.sigmoid(
            self.fc2(w)
        )

        return x * w


class ConvNeXtRefine(nn.Module):
    def __init__(
        self,
        channels,
        drop=0.0,
    ):
        super().__init__()

        self.dw = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = make_group_norm(
            channels
        )

        self.pw1 = nn.Conv2d(
            channels,
            channels * 4,
            kernel_size=1,
        )

        self.pw2 = nn.Conv2d(
            channels * 4,
            channels,
            kernel_size=1,
        )

        self.drop = nn.Dropout2d(
            drop
        )

        self.se = SEBlock(
            channels
        )

    def forward(self, x):
        residual = x

        x = self.dw(x)

        x = self.norm(x)

        x = F.gelu(
            self.pw1(x)
        )

        x = self.drop(
            self.pw2(x)
        )

        x = self.se(x)

        return residual + x


class ModalityProjector(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        out_size,
    ):
        super().__init__()

        self.out_size = out_size

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
            ConvNeXtRefine(
                out_channels,
                drop=0.0,
            ),
        )

    def forward(self, x):
        x = self.proj(x)

        if (
            x.shape[-2:]
            != (
                self.out_size,
                self.out_size,
            )
        ):
            x = F.interpolate(
                x,
                size=(
                    self.out_size,
                    self.out_size,
                ),
                mode="bilinear",
                align_corners=False,
            )

        return x


class ReliabilityRouter(nn.Module):
    """
    Spatial SAR-optical router.

    P(clear) is used as a prior on the optical route,
    while the learned router can still override that prior.
    """

    def __init__(
        self,
        channels,
        hidden=64,
        prior_strength=0.50,
    ):
        super().__init__()

        hidden = min(
            hidden,
            max(
                channels // 2,
                16,
            ),
        )

        self.prior_strength = (
            float(
                prior_strength
            )
        )

        self.net = nn.Sequential(
            nn.Conv2d(
                channels * 2 + 1,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                hidden
            ),
            nn.GELU(),
            nn.Conv2d(
                hidden,
                2,
                kernel_size=1,
            ),
        )

        # Start near neutral learned routing.
        nn.init.zeros_(
            self.net[-1].weight
        )

        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        r = F.interpolate(
            pclear,
            size=sar.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        r = r.clamp(
            0.0,
            1.0,
        )

        logits = self.net(
            torch.cat(
                [
                    sar,
                    optical,
                    r,
                ],
                dim=1,
            )
        )

        # Optical prior. Clamp avoids -inf at P(clear)=0.
        logits[
            :,
            1:2,
        ] = (
            logits[
                :,
                1:2,
            ]
            + self.prior_strength
            * torch.log(
                r.clamp(
                    min=0.05,
                    max=1.0,
                )
            )
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        w_sar = weights[
            :,
            0:1,
        ]

        w_opt = weights[
            :,
            1:2,
        ]

        fused = (
            w_sar * sar
            + w_opt * optical
        )

        return (
            fused,
            weights,
        )


class DeepRIFT(nn.Module):
    """
    Zero-initialized reliability-conditioned cross attention.

    When P(clear) is high:
      SAR can borrow more optical context.

    When P(clear) is low:
      optical can borrow more SAR context.

    Output projections start at zero, so the block is an
    exact identity at initialization.
    """

    def __init__(
        self,
        channels=512,
        heads=8,
        dropout=0.0,
    ):
        super().__init__()

        self.channels = channels

        self.sar_norm = nn.LayerNorm(
            channels
        )

        self.opt_norm = nn.LayerNorm(
            channels
        )

        self.sar_from_opt = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.opt_from_sar = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.sar_out = nn.Linear(
            channels,
            channels,
        )

        self.opt_out = nn.Linear(
            channels,
            channels,
        )

        nn.init.zeros_(
            self.sar_out.weight
        )

        nn.init.zeros_(
            self.sar_out.bias
        )

        nn.init.zeros_(
            self.opt_out.weight
        )

        nn.init.zeros_(
            self.opt_out.bias
        )

    @staticmethod
    def _flatten(x):
        return (
            x.flatten(2)
            .transpose(1, 2)
            .contiguous()
        )

    @staticmethod
    def _unflatten(
        tokens,
        height,
        width,
    ):
        return (
            tokens
            .transpose(1, 2)
            .reshape(
                tokens.shape[0],
                tokens.shape[2],
                height,
                width,
            )
            .contiguous()
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        b, c, h, w = sar.shape

        sar_t = self._flatten(
            sar
        )

        opt_t = self._flatten(
            optical
        )

        r = F.adaptive_avg_pool2d(
            pclear,
            output_size=(h, w),
        )

        r_t = (
            r.flatten(2)
            .transpose(1, 2)
            .clamp(0.0, 1.0)
        )

        sar_n = self.sar_norm(
            sar_t
        )

        opt_n = self.opt_norm(
            opt_t
        )

        # Clear optical context receives larger K/V magnitude.
        opt_context = (
            opt_n
            * (
                0.25
                + 0.75 * r_t
            )
        )

        sar_delta, _ = (
            self.sar_from_opt(
                query=sar_n,
                key=opt_context,
                value=opt_context,
                need_weights=False,
            )
        )

        # Under low P(clear), the optical branch borrows SAR more.
        opt_delta, _ = (
            self.opt_from_sar(
                query=opt_n,
                key=sar_n,
                value=sar_n,
                need_weights=False,
            )
        )

        sar_t = (
            sar_t
            + r_t
            * self.sar_out(
                sar_delta
            )
        )

        opt_t = (
            opt_t
            + (1.0 - r_t)
            * self.opt_out(
                opt_delta
            )
        )

        sar = self._unflatten(
            sar_t,
            h,
            w,
        )

        optical = self._unflatten(
            opt_t,
            h,
            w,
        )

        return sar, optical


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.fuse = nn.Sequential(
            ConvNormAct(
                in_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.fuse(x)


class FloodShiftTerraMindRIFT(nn.Module):
    def __init__(
        self,
        num_classes=3,
        pretrained_backbone=False,
    ):
        super().__init__()

        # concat is used deliberately:
        # each spatial token contains S2 and S1 embeddings
        # concatenated in the modality order supplied below.
        self.backbone = (
            BACKBONE_REGISTRY.build(
                "terramind_v1_base",
                pretrained=pretrained_backbone,
                img_size=TILE_SIZE,
                modalities=[
                    "S2L1C",
                    "S1GRD",
                ],
                bands={
                    "S2L1C":
                        TM_OPTICAL_BAND_NAMES,
                },
                merge_method="concat",
            )
        )

        self.register_buffer(
            "project_opt_mean",
            torch.tensor(
                OPTICAL_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_opt_std",
            torch.tensor(
                OPTICAL_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_mean",
            torch.tensor(
                SAR_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_std",
            torch.tensor(
                SAR_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_mean",
            torch.tensor(
                TM_S2_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_std",
            torch.tensor(
                TM_S2_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_mean",
            torch.tensor(
                TM_S1_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_std",
            torch.tensor(
                TM_S1_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.sar_projectors = (
            nn.ModuleList()
        )

        self.opt_projectors = (
            nn.ModuleList()
        )

        self.routers = nn.ModuleList()

        for (
            channels,
            out_size,
        ) in zip(
            PYRAMID_CHANNELS,
            PYRAMID_SIZES,
        ):
            self.sar_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.opt_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.routers.append(
                ReliabilityRouter(
                    channels,
                    prior_strength=0.50,
                )
            )

        self.deep_rift = DeepRIFT(
            channels=PYRAMID_CHANNELS[-1],
            heads=8,
            dropout=0.0,
        )

        self.deep_context = nn.Sequential(
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
        )

        self.decoder3 = DecoderBlock(
            512,
            256,
            256,
        )

        self.decoder2 = DecoderBlock(
            256,
            128,
            128,
        )

        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
        )

        self.final_refine = nn.Sequential(
            ConvNormAct(
                64,
                64,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                64,
                drop=0.05,
            ),
        )

        self.segmentation_head = (
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            )
        )

        self.aux_head = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(64),
            nn.GELU(),
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            ),
        )

    def to_terramind_inputs(
        self,
        sar_z,
        optical_z,
    ):
        # Detect frozen invalid-fill pixels BEFORE inverse affine.
        opt_invalid = (
            optical_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        sar_invalid = (
            sar_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        # Frozen optical z-score -> reflectance -> DN.
        optical_reflectance = (
            optical_z
            * self.project_opt_std
            + self.project_opt_mean
        )

        optical_dn = (
            optical_reflectance
            * 10000.0
        )

        optical_tm = (
            optical_dn
            - self.tm_opt_mean
        ) / self.tm_opt_std

        # Frozen SAR z-score -> raw dB.
        sar_db = (
            sar_z
            * self.project_sar_std
            + self.project_sar_mean
        )

        sar_tm = (
            sar_db
            - self.tm_sar_mean
        ) / self.tm_sar_std

        # Preserve invalid-fill convention in model space.
        optical_tm = optical_tm.masked_fill(
            opt_invalid,
            0.0,
        )

        sar_tm = sar_tm.masked_fill(
            sar_invalid,
            0.0,
        )

        return (
            sar_tm,
            optical_tm,
        )

    @staticmethod
    def tokens_to_map(tokens):
        b, n, c = tokens.shape

        side = int(
            math.isqrt(n)
        )

        if (
            side * side
            != n
        ):
            raise RuntimeError(
                f"TerraMind token count {n} "
                "is not a square grid."
            )

        return (
            tokens
            .transpose(1, 2)
            .reshape(
                b,
                c,
                side,
                side,
            )
            .contiguous()
        )

    def extract_modality_tokens(
        self,
        tm_outputs,
    ):
        if (
            not isinstance(
                tm_outputs,
                (list, tuple),
            )
            or len(tm_outputs) < 12
        ):
            raise RuntimeError(
                "Unexpected TerraMind output. "
                "Expected list of 12 transformer-layer outputs."
            )

        sar_tokens = []
        opt_tokens = []

        for layer_index in TM_LAYER_INDICES:
            tokens = tm_outputs[
                layer_index
            ]

            if (
                not torch.is_tensor(tokens)
                or tokens.ndim != 3
            ):
                raise RuntimeError(
                    "TerraMind concat output should be "
                    "[B, tokens, embedding]."
                )

            expected_dim = (
                TM_EMBED_DIM * 2
            )

            if (
                tokens.shape[-1]
                != expected_dim
            ):
                raise RuntimeError(
                    "Expected concat embedding dimension "
                    f"{expected_dim}, got "
                    f"{tokens.shape[-1]}. "
                    "Check TerraTorch version and merge_method."
                )

            # modalities were supplied in order:
            # [S2L1C, S1GRD]
            optical_part = tokens[
                :,
                :,
                :TM_EMBED_DIM,
            ]

            sar_part = tokens[
                :,
                :,
                TM_EMBED_DIM:,
            ]

            opt_tokens.append(
                optical_part
            )

            sar_tokens.append(
                sar_part
            )

        return (
            sar_tokens,
            opt_tokens,
        )

    def forward(
        self,
        sar,
        optical,
        reliability,
        return_aux=False,
    ):
        (
            sar_tm,
            optical_tm,
        ) = self.to_terramind_inputs(
            sar,
            optical,
        )

        tm_outputs = self.backbone(
            {
                "S2L1C": optical_tm,
                "S1GRD": sar_tm,
            }
        )

        (
            sar_tokens,
            opt_tokens,
        ) = self.extract_modality_tokens(
            tm_outputs
        )

        sar_maps = []
        opt_maps = []

        for i in range(4):
            sar_map = (
                self.tokens_to_map(
                    sar_tokens[i]
                )
            )

            opt_map = (
                self.tokens_to_map(
                    opt_tokens[i]
                )
            )

            sar_map = (
                self.sar_projectors[i](
                    sar_map
                )
            )

            opt_map = (
                self.opt_projectors[i](
                    opt_map
                )
            )

            sar_maps.append(
                sar_map
            )

            opt_maps.append(
                opt_map
            )

        # RIFT only at deepest 16x16 level.
        (
            sar_maps[-1],
            opt_maps[-1],
        ) = self.deep_rift(
            sar_maps[-1],
            opt_maps[-1],
            reliability,
        )

        fused = []
        router_weights = []

        for i in range(4):
            (
                fused_i,
                weights_i,
            ) = self.routers[i](
                sar_maps[i],
                opt_maps[i],
                reliability,
            )

            fused.append(
                fused_i
            )

            router_weights.append(
                weights_i
            )

        f0, f1, f2, f3 = fused

        x = self.deep_context(
            f3
        )

        x = self.decoder3(
            x,
            f2,
        )

        x = self.decoder2(
            x,
            f1,
        )

        aux_logits = self.aux_head(
            x
        )

        x = self.decoder1(
            x,
            f0,
        )

        x = F.interpolate(
            x,
            size=(
                TILE_SIZE,
                TILE_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(
            x
        )

        logits = (
            self.segmentation_head(
                x
            )
        )

        if return_aux:
            return {
                "logits": logits,
                "aux_logits": aux_logits,
                "router_weights": router_weights,
            }

        return logits

In [ ]:
# ============================================================
# CELL: P14-C07 — REBUILD EXACT ACCEPTED 04H-CAL MODEL
# ============================================================
class HierarchicalResidualHead(nn.Module):
    def __init__(self, in_channels=16, hidden=48, max_delta=1.50):
        super().__init__()
        self.max_delta = float(max_delta)
        self.net = nn.Sequential(
            ConvNormAct(in_channels, hidden, 3),
            ConvNeXtRefine(hidden, drop=0.05),
            ConvNormAct(hidden, hidden, 3),
            nn.Conv2d(hidden, 2, kernel_size=1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, base_prob, sar, optical, reliability):
        x = torch.cat([base_prob, sar, optical, reliability], dim=1)
        raw = self.net(x).float()
        return self.max_delta * torch.tanh(raw / self.max_delta)

class FloodShift04H(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.head = HierarchicalResidualHead()

        for p in self.base.parameters():
            p.requires_grad = False
        self.base.eval()

    @property
    def backbone(self):
        return self.base.backbone

    def train(self, mode=True):
        super().train(mode)
        self.base.eval()
        self.head.train(mode)
        return self

    def forward(self, sar, optical, reliability, return_aux=False):
        with torch.no_grad():
            base_out = self.base(
                sar,
                optical,
                reliability,
                return_aux=True,
            )
            base_logits = base_out["logits"].float()
            base_prob = torch.softmax(base_logits, dim=1)

        correction = self.head(
            base_prob.to(sar.dtype),
            sar,
            optical,
            reliability,
        )

        delta_water = correction[:, 0]
        delta_type = correction[:, 1]

        base_water_logit = (
            torch.logsumexp(base_logits[:, 1:3], dim=1)
            - base_logits[:, 0]
        )
        base_type_logit = base_logits[:, 2] - base_logits[:, 1]

        water_logit = base_water_logit + delta_water
        type_logit = base_type_logit + delta_type

        p_water = torch.sigmoid(water_logit)
        p_temp_given_water = torch.sigmoid(type_logit)

        p0 = 1.0 - p_water
        p1 = p_water * (1.0 - p_temp_given_water)
        p2 = p_water * p_temp_given_water

        probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
        log_probs = probs.log()

        if return_aux:
            return {
                "logits": log_probs,
                "probs": probs,
                "base_logits": base_logits,
                "base_prob": base_prob,
                "base_water_logit": base_water_logit,
                "base_type_logit": base_type_logit,
                "water_logit": water_logit,
                "type_logit": type_logit,
                "delta_water": delta_water,
                "delta_type": delta_type,
            }

        return log_probs

payload = torch.load(CHECKPOINT_04H, map_location="cpu", weights_only=False)

assert payload.get("base_sha256") == EXPECTED_04E_SHA256
assert int(payload.get("epoch", -1)) == 4
assert payload.get("tag") == "best_mIoU"

base = FloodShiftTerraMindRIFT(
    num_classes=NUM_CLASSES,
    pretrained_backbone=False,
)
model = FloodShift04H(base)

missing, unexpected = model.load_state_dict(
    payload["state_dict"],
    strict=False,
)

assert not missing, f"Missing keys: {missing[:10]}"
assert not unexpected, f"Unexpected keys: {unexpected[:10]}"

bad = [
    n for n, t in model.state_dict().items()
    if torch.is_floating_point(t) and not torch.isfinite(t).all()
]
assert not bad

model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad = False

print("=" * 112)
print("P14-C07 — ACCEPTED 04H-CAL MODEL READY")
print("=" * 112)
print("Epoch          :", payload["epoch"])
print("Tag            :", payload["tag"])
print("Type bias      :", TYPE_BIAS)
print("Temperature    :", FROZEN_TEMPERATURE)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("P14-C07 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P14-C08 — METRICS + FROZEN 4-WAY TTA + COUNTERFACTUALS
# ============================================================
TTA_MODES = ["identity", "hflip", "vflip", "rot180"]

def confusion_from_predictions(pred, target):
    valid = target != IGNORE_INDEX
    target_v = target[valid].long()
    pred_v = pred[valid].long()

    indices = NUM_CLASSES * target_v + pred_v
    return torch.bincount(
        indices,
        minlength=NUM_CLASSES * NUM_CLASSES,
    ).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics_from_confusion(cm):
    if torch.is_tensor(cm):
        cm = cm.detach().cpu().numpy().astype(np.float64)
    else:
        cm = np.asarray(cm, dtype=np.float64)

    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    den_iou = tp + fp + fn
    iou = np.divide(
        tp,
        den_iou,
        out=np.full_like(tp, np.nan),
        where=den_iou > 0,
    )

    accuracy = float(tp.sum() / max(cm.sum(), 1.0))

    out = {
        "mIoU": float(np.nanmean(iou)),
        "accuracy": accuracy,
        "confusion": cm.tolist(),
    }

    for i, name in enumerate(CLASS_NAMES):
        out[f"{name}_IoU"] = float(iou[i])

    return out

def compact_metrics(m):
    return {
        "mIoU": round(m["mIoU"], 4),
        "non_flood": round(m["non_flood_IoU"], 4),
        "permanent": round(m["permanent_water_IoU"], 4),
        "temporary": round(m["temporary_flood_IoU"], 4),
    }

def _tta_transform(x, mode):
    if mode == "identity":
        return x
    if mode == "hflip":
        return torch.flip(x, dims=[-1])
    if mode == "vflip":
        return torch.flip(x, dims=[-2])
    if mode == "rot180":
        return torch.rot90(x, k=2, dims=(-2, -1))
    raise ValueError(mode)

def _tta_inverse(x, mode):
    return _tta_transform(x, mode)

def calibrated_log_probs_from_aux(aux):
    water_logit = aux["water_logit"].float()
    type_logit = aux["type_logit"].float() + TYPE_BIAS

    p_water = torch.sigmoid(water_logit)
    p_temp_given_water = torch.sigmoid(type_logit)

    p0 = 1.0 - p_water
    p1 = p_water * (1.0 - p_temp_given_water)
    p2 = p_water * p_temp_given_water

    probs = torch.stack([p0, p1, p2], dim=1).clamp_min(1e-7)
    return probs.log()

@torch.inference_mode()
def tta_log_probs_variant(model_to_eval, sar, optical, reliability, variant):
    if variant == "full":
        sar_v = sar
        opt_v = optical
        rel_v = reliability

    elif variant == "no_sar":
        sar_v = torch.zeros_like(sar)
        opt_v = optical
        rel_v = reliability

    elif variant == "no_optical":
        sar_v = sar
        opt_v = torch.zeros_like(optical)
        rel_v = torch.zeros_like(reliability)

    else:
        raise ValueError(variant)

    total = None

    for mode in TTA_MODES:
        sar_t = _tta_transform(sar_v, mode)
        opt_t = _tta_transform(opt_v, mode)
        rel_t = _tta_transform(rel_v, mode)

        with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
            aux = model_to_eval(
                sar_t,
                opt_t,
                rel_t,
                return_aux=True,
            )

        lp = calibrated_log_probs_from_aux(aux)
        lp = _tta_inverse(lp, mode).float()

        total = lp if total is None else total + lp

    return total / len(TTA_MODES)

def probs_with_temperature(log_probs):
    return torch.softmax(
        log_probs.float() / FROZEN_TEMPERATURE,
        dim=1,
    )

def normalized_entropy(probs):
    return (
        -(probs * probs.clamp_min(1e-8).log()).sum(dim=1)
        / math.log(NUM_CLASSES)
    )

def normalized_js(p, q):
    """
    Jensen-Shannon divergence normalized by ln(2), so output is in [0,1].
    Shapes: [B,C,H,W] -> [B,H,W]
    """
    p = p.clamp_min(1e-8)
    q = q.clamp_min(1e-8)
    m = 0.5 * (p + q)

    kl_pm = (p * (p.log() - m.log())).sum(dim=1)
    kl_qm = (q * (q.log() - m.log())).sum(dim=1)

    js = 0.5 * (kl_pm + kl_qm)
    return (js / math.log(2.0)).clamp(0.0, 1.0)

print("P14-C08 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P14-C09 — COUNTERFACTUAL PASS
#
# Runs full / no-SAR / no-optical on both internal val halves.
# Stores only compact per-tile evidence, not giant pixel dumps.
# ============================================================
def evaluate_counterfactual_loader(loader, split_name):
    cm_full = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
    cm_no_sar = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
    cm_no_opt = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)

    tile_rows = []

    pixel_error_labels = []
    pixel_entropy_scores = []
    pixel_fragility_scores = []

    model.eval()
    t0 = time.time()

    with torch.inference_mode():
        for bi, batch in enumerate(loader, start=1):
            sar = batch["sar"].to(DEVICE, non_blocking=True)
            opt = batch["optical"].to(DEVICE, non_blocking=True)
            rel = batch["reliability"].to(DEVICE, non_blocking=True)
            y = batch["target"].to(DEVICE, non_blocking=True)

            lp_full = tta_log_probs_variant(model, sar, opt, rel, "full")
            lp_no_sar = tta_log_probs_variant(model, sar, opt, rel, "no_sar")
            lp_no_opt = tta_log_probs_variant(model, sar, opt, rel, "no_optical")

            p_full = probs_with_temperature(lp_full)
            p_no_sar = probs_with_temperature(lp_no_sar)
            p_no_opt = probs_with_temperature(lp_no_opt)

            pred_full = p_full.argmax(1)
            pred_no_sar = p_no_sar.argmax(1)
            pred_no_opt = p_no_opt.argmax(1)

            cm_full += confusion_from_predictions(pred_full, y).cpu()
            cm_no_sar += confusion_from_predictions(pred_no_sar, y).cpu()
            cm_no_opt += confusion_from_predictions(pred_no_opt, y).cpu()

            js_sar = normalized_js(p_full, p_no_sar)
            js_opt = normalized_js(p_full, p_no_opt)

            fragility = torch.maximum(js_sar, js_opt)
            entropy = normalized_entropy(p_full)

            full_conf = p_full.max(dim=1).values

            # Signed support for the ORIGINAL full-model predicted class.
            full_class = pred_full[:, None]
            p_full_class = p_full.gather(1, full_class).squeeze(1)
            p_no_sar_class = p_no_sar.gather(1, full_class).squeeze(1)
            p_no_opt_class = p_no_opt.gather(1, full_class).squeeze(1)

            support_sar = p_full_class - p_no_sar_class
            support_opt = p_full_class - p_no_opt_class

            valid = y != IGNORE_INDEX
            err = (pred_full != y) & valid

            # Compact pixel samples for AUROC/AUPRC only.
            # Full arrays are manageable on 34 scenes, but we subsample deterministically.
            for i in range(y.shape[0]):
                v = valid[i].flatten()
                idx = torch.nonzero(v, as_tuple=False).squeeze(1)

                if idx.numel() > 0:
                    take = min(8192, idx.numel())
                    # deterministic evenly-spaced selection; no random fitting
                    pos = torch.linspace(
                        0,
                        idx.numel() - 1,
                        steps=take,
                        device=idx.device,
                    ).long()
                    chosen = idx[pos]

                    pixel_error_labels.append(
                        err[i].flatten()[chosen].detach().cpu().numpy().astype(np.uint8)
                    )
                    pixel_entropy_scores.append(
                        entropy[i].flatten()[chosen].detach().cpu().numpy().astype(np.float32)
                    )
                    pixel_fragility_scores.append(
                        fragility[i].flatten()[chosen].detach().cpu().numpy().astype(np.float32)
                    )

            scene_ids = list(batch["scene_id"])
            tile_indices = batch["tile_index"]

            for i in range(y.shape[0]):
                v = valid[i]
                if not v.any():
                    continue

                pred_water = v & ((pred_full[i] == 1) | (pred_full[i] == 2))
                pred_temp = v & (pred_full[i] == 2)
                pred_perm = v & (pred_full[i] == 1)

                mean_js_sar = float(js_sar[i][v].mean().cpu())
                mean_js_opt = float(js_opt[i][v].mean().cpu())
                total_attr = mean_js_sar + mean_js_opt
                sar_share = mean_js_sar / max(total_attr, 1e-12)
                opt_share = mean_js_opt / max(total_attr, 1e-12)

                if pred_water.any():
                    water_js_sar = float(js_sar[i][pred_water].mean().cpu())
                    water_js_opt = float(js_opt[i][pred_water].mean().cpu())
                else:
                    water_js_sar = float("nan")
                    water_js_opt = float("nan")

                if pred_temp.any():
                    temp_js_sar = float(js_sar[i][pred_temp].mean().cpu())
                    temp_js_opt = float(js_opt[i][pred_temp].mean().cpu())
                else:
                    temp_js_sar = float("nan")
                    temp_js_opt = float("nan")

                if pred_perm.any():
                    perm_js_sar = float(js_sar[i][pred_perm].mean().cpu())
                    perm_js_opt = float(js_opt[i][pred_perm].mean().cpu())
                else:
                    perm_js_sar = float("nan")
                    perm_js_opt = float("nan")

                tile_rows.append({
                    "split": split_name,
                    "scene_id": str(scene_ids[i]),
                    "tile_index": int(tile_indices[i]),
                    "valid_pixels": int(v.sum().cpu()),
                    "error_rate": float(((pred_full[i] != y[i]) & v).sum().cpu() / v.sum().cpu()),
                    "mean_pclear": float(rel[i, 0][v].mean().cpu()),
                    "mean_entropy": float(entropy[i][v].mean().cpu()),
                    "mean_confidence": float(full_conf[i][v].mean().cpu()),
                    "mean_js_sar": mean_js_sar,
                    "mean_js_optical": mean_js_opt,
                    "mean_fragility": float(fragility[i][v].mean().cpu()),
                    "sar_share": sar_share,
                    "optical_share": opt_share,
                    "flip_rate_no_sar": float((pred_no_sar[i][v] != pred_full[i][v]).float().mean().cpu()),
                    "flip_rate_no_optical": float((pred_no_opt[i][v] != pred_full[i][v]).float().mean().cpu()),
                    "support_sar": float(support_sar[i][v].mean().cpu()),
                    "support_optical": float(support_opt[i][v].mean().cpu()),
                    "water_js_sar": water_js_sar,
                    "water_js_optical": water_js_opt,
                    "temporary_js_sar": temp_js_sar,
                    "temporary_js_optical": temp_js_opt,
                    "permanent_js_sar": perm_js_sar,
                    "permanent_js_optical": perm_js_opt,
                    "pred_nonflood_fraction": float((pred_full[i][v] == 0).float().mean().cpu()),
                    "pred_permanent_fraction": float((pred_full[i][v] == 1).float().mean().cpu()),
                    "pred_temporary_fraction": float((pred_full[i][v] == 2).float().mean().cpu()),
                })

            if bi == 1 or bi % 20 == 0 or bi == len(loader):
                print(
                    f"[{bi:03d}/{len(loader)}] {split_name} "
                    f"full / -SAR / -OPT"
                )

    labels = np.concatenate(pixel_error_labels)
    entropy_scores = np.concatenate(pixel_entropy_scores)
    fragility_scores = np.concatenate(pixel_fragility_scores)

    combined_scores = 0.5 * entropy_scores + 0.5 * fragility_scores

    def safe_auc(scores):
        if np.unique(labels).size < 2:
            return float("nan"), float("nan")
        return (
            float(roc_auc_score(labels, scores)),
            float(average_precision_score(labels, scores)),
        )

    ent_auroc, ent_auprc = safe_auc(entropy_scores)
    frag_auroc, frag_auprc = safe_auc(fragility_scores)
    comb_auroc, comb_auprc = safe_auc(combined_scores)

    return {
        "full_metrics": metrics_from_confusion(cm_full),
        "no_sar_metrics": metrics_from_confusion(cm_no_sar),
        "no_optical_metrics": metrics_from_confusion(cm_no_opt),
        "tile_df": pd.DataFrame(tile_rows),
        "error_detection": {
            "entropy_AUROC": ent_auroc,
            "entropy_AUPRC": ent_auprc,
            "fragility_AUROC": frag_auroc,
            "fragility_AUPRC": frag_auprc,
            "combined_AUROC": comb_auroc,
            "combined_AUPRC": comb_auprc,
        },
        "minutes": (time.time() - t0) / 60.0,
    }

print("Running calibration-half counterfactual pass...")
cal_result = evaluate_counterfactual_loader(
    val_cal_loader,
    "calibration",
)

print("\nRunning attribution-half counterfactual pass...")
attr_result = evaluate_counterfactual_loader(
    val_attr_loader,
    "attribution",
)

print("P14-C09 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P14-C10 — MANDATORY COUNTERFACTUAL INTEGRITY CHECK
#
# Attribution half is exactly the same 34-scene half used for
# Notebook-05 robustness evaluation.
# Therefore these three mIoUs should reproduce the earlier:
# clean, SAR-missing, optical-missing conditions.
# ============================================================
EXPECTED_ATTR = {
    "full_mIoU": 0.8265,
    "no_sar_mIoU": 0.5609,
    "no_optical_mIoU": 0.2917,
}

print("=" * 112)
print("P14-C10 — COUNTERFACTUAL INTEGRITY")
print("=" * 112)

print("FULL      :", compact_metrics(attr_result["full_metrics"]))
print("NO SAR    :", compact_metrics(attr_result["no_sar_metrics"]))
print("NO OPTICAL:", compact_metrics(attr_result["no_optical_metrics"]))

assert abs(
    attr_result["full_metrics"]["mIoU"]
    - EXPECTED_ATTR["full_mIoU"]
) < 0.0020

assert abs(
    attr_result["no_sar_metrics"]["mIoU"]
    - EXPECTED_ATTR["no_sar_mIoU"]
) < 0.0020

assert abs(
    attr_result["no_optical_metrics"]["mIoU"]
    - EXPECTED_ATTR["no_optical_mIoU"]
) < 0.0020

print()
print("✅ full path matches Notebook-05 clean robustness-half result")
print("✅ no-SAR path matches Notebook-05 SAR-missing result")
print("✅ no-optical path matches Notebook-05 optical-missing result")
print("P14-C10 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P14-C11 — FREEZE ATTRIBUTION FLAGS ON CALIBRATION HALF
# ============================================================
cal_tiles = cal_result["tile_df"].copy()
attr_tiles = attr_result["tile_df"].copy()

# High fragility is a diagnostic flag, not a learned classifier.
# Threshold frozen from calibration-half tile distribution.
HIGH_FRAGILITY_THRESHOLD = float(
    np.quantile(cal_tiles["mean_fragility"].to_numpy(), 0.90)
)

# Dominance is reported only as a readable summary.
DOMINANCE_SHARE = 0.65

def attribution_label(row):
    if row["sar_share"] >= DOMINANCE_SHARE:
        return "SAR-dominant"
    if row["optical_share"] >= DOMINANCE_SHARE:
        return "Optical-dominant"
    return "Balanced"

cal_tiles["dominance"] = cal_tiles.apply(attribution_label, axis=1)
attr_tiles["dominance"] = attr_tiles.apply(attribution_label, axis=1)

cal_tiles["high_fragility"] = (
    cal_tiles["mean_fragility"] >= HIGH_FRAGILITY_THRESHOLD
)
attr_tiles["high_fragility"] = (
    attr_tiles["mean_fragility"] >= HIGH_FRAGILITY_THRESHOLD
)

print("=" * 112)
print("P14-C11 — FROZEN ATTRIBUTION FLAGS")
print("=" * 112)
print("90th-percentile calibration fragility threshold:", HIGH_FRAGILITY_THRESHOLD)
print("Dominance share threshold:", DOMINANCE_SHARE)
print()
print("Attribution-half dominance counts:")
print(attr_tiles["dominance"].value_counts())
print()
print("Attribution-half high-fragility tile rate:",
      float(attr_tiles["high_fragility"].mean()))
print("P14-C11 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P14-C12 — SENSOR CONTRIBUTION ANALYSIS
# ============================================================
def weighted_mean(df, value_col, weight_col="valid_pixels"):
    d = df[[value_col, weight_col]].dropna()
    if len(d) == 0:
        return float("nan")
    w = d[weight_col].to_numpy(dtype=np.float64)
    x = d[value_col].to_numpy(dtype=np.float64)
    return float(np.sum(w * x) / max(np.sum(w), 1e-12))

summary = {
    "mean_js_sar": weighted_mean(attr_tiles, "mean_js_sar"),
    "mean_js_optical": weighted_mean(attr_tiles, "mean_js_optical"),
    "mean_fragility": weighted_mean(attr_tiles, "mean_fragility"),
    "sar_share": weighted_mean(attr_tiles, "sar_share"),
    "optical_share": weighted_mean(attr_tiles, "optical_share"),
    "flip_rate_no_sar": weighted_mean(attr_tiles, "flip_rate_no_sar"),
    "flip_rate_no_optical": weighted_mean(attr_tiles, "flip_rate_no_optical"),
    "support_sar": weighted_mean(attr_tiles, "support_sar"),
    "support_optical": weighted_mean(attr_tiles, "support_optical"),
    "water_js_sar": weighted_mean(attr_tiles, "water_js_sar"),
    "water_js_optical": weighted_mean(attr_tiles, "water_js_optical"),
    "temporary_js_sar": weighted_mean(attr_tiles, "temporary_js_sar"),
    "temporary_js_optical": weighted_mean(attr_tiles, "temporary_js_optical"),
    "permanent_js_sar": weighted_mean(attr_tiles, "permanent_js_sar"),
    "permanent_js_optical": weighted_mean(attr_tiles, "permanent_js_optical"),
}

rho_opt, p_opt = spearmanr(
    attr_tiles["mean_pclear"],
    attr_tiles["optical_share"],
    nan_policy="omit",
)

rho_sar, p_sar = spearmanr(
    attr_tiles["mean_pclear"],
    attr_tiles["sar_share"],
    nan_policy="omit",
)

summary["spearman_pclear_vs_optical_share"] = float(rho_opt)
summary["spearman_pclear_vs_optical_share_p"] = float(p_opt)
summary["spearman_pclear_vs_sar_share"] = float(rho_sar)
summary["spearman_pclear_vs_sar_share_p"] = float(p_sar)

print("=" * 112)
print("P14-C12 — ATTRIBUTION SUMMARY")
print("=" * 112)
for k, v in summary.items():
    print(f"{k:<42}: {v:.6f}")

print()
print("Error-detection diagnostics:")
for k, v in attr_result["error_detection"].items():
    print(f"{k:<24}: {v:.6f}")

print()
if summary["mean_js_optical"] > summary["mean_js_sar"]:
    print("Overall model dependence is stronger on OPTICAL than SAR.")
else:
    print("Overall model dependence is stronger on SAR than OPTICAL.")

if summary["temporary_js_optical"] > summary["temporary_js_sar"]:
    print("Temporary-flood predictions are more sensitive to OPTICAL removal.")
else:
    print("Temporary-flood predictions are more sensitive to SAR removal.")

print("P14-C12 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P14-C13 — SCENE-LEVEL GROUNDED ATTRIBUTION EVIDENCE
# ============================================================
WEIGHTED_COLS = [
    "error_rate",
    "mean_pclear",
    "mean_entropy",
    "mean_confidence",
    "mean_js_sar",
    "mean_js_optical",
    "mean_fragility",
    "sar_share",
    "optical_share",
    "flip_rate_no_sar",
    "flip_rate_no_optical",
    "support_sar",
    "support_optical",
    "water_js_sar",
    "water_js_optical",
    "temporary_js_sar",
    "temporary_js_optical",
    "permanent_js_sar",
    "permanent_js_optical",
    "pred_nonflood_fraction",
    "pred_permanent_fraction",
    "pred_temporary_fraction",
]

scene_rows = []

for scene_id, g in attr_tiles.groupby("scene_id"):
    row = {
        "scene_id": scene_id,
        "tiles": int(len(g)),
        "valid_pixels": int(g["valid_pixels"].sum()),
    }

    for col in WEIGHTED_COLS:
        row[col] = weighted_mean(g, col)

    row["dominance"] = attribution_label(row)
    row["high_fragility_tile_fraction"] = float(g["high_fragility"].mean())

    scene_rows.append(row)

scene_df = pd.DataFrame(scene_rows)

scene_df = scene_df.sort_values(
    ["mean_fragility", "scene_id"],
    ascending=[False, True],
).reset_index(drop=True)

print("=" * 112)
print("P14-C13 — SCENE EVIDENCE")
print("=" * 112)
print("Scenes:", len(scene_df))
print()
print("Top 10 scenes by counterfactual fragility:")
display(
    scene_df[
        [
            "scene_id",
            "mean_pclear",
            "mean_fragility",
            "sar_share",
            "optical_share",
            "flip_rate_no_sar",
            "flip_rate_no_optical",
            "pred_temporary_fraction",
            "dominance",
            "high_fragility_tile_fraction",
        ]
    ].head(10)
)

print("P14-C13 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P14-C14 — CLASS-SPECIFIC ATTRIBUTION TABLE
# ============================================================
class_attr = pd.DataFrame([
    {
        "prediction_region": "all_valid_pixels",
        "SAR_JS": summary["mean_js_sar"],
        "Optical_JS": summary["mean_js_optical"],
    },
    {
        "prediction_region": "predicted_water",
        "SAR_JS": summary["water_js_sar"],
        "Optical_JS": summary["water_js_optical"],
    },
    {
        "prediction_region": "predicted_permanent_water",
        "SAR_JS": summary["permanent_js_sar"],
        "Optical_JS": summary["permanent_js_optical"],
    },
    {
        "prediction_region": "predicted_temporary_flood",
        "SAR_JS": summary["temporary_js_sar"],
        "Optical_JS": summary["temporary_js_optical"],
    },
])

class_attr["dominant_sensor"] = np.where(
    class_attr["SAR_JS"] > class_attr["Optical_JS"],
    "SAR",
    "Optical",
)

class_attr["Optical_minus_SAR"] = (
    class_attr["Optical_JS"] - class_attr["SAR_JS"]
)

print("=" * 112)
print("P14-C14 — CLASS-SPECIFIC COUNTERFACTUAL ATTRIBUTION")
print("=" * 112)
display(class_attr)
print("P14-C14 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P14-C15 — SAVE EVIDENCE + PROTOCOL
# ============================================================
cal_tile_path = EVIDENCE_ROOT / "FloodShift_06_calibration_tile_attribution.csv"
attr_tile_path = EVIDENCE_ROOT / "FloodShift_06_attribution_tile_evidence.csv"
scene_path = EVIDENCE_ROOT / "FloodShift_06_scene_evidence.csv"
class_path = EVIDENCE_ROOT / "FloodShift_06_class_attribution.csv"

cal_tiles.to_csv(cal_tile_path, index=False)
attr_tiles.to_csv(attr_tile_path, index=False)
scene_df.to_csv(scene_path, index=False)
class_attr.to_csv(class_path, index=False)

protocol = {
    "experiment": "FloodShift-06-Counterfactual-Sensor-Attribution",
    "model": "accepted 04H-CAL",
    "source_04h_sha256": SHA_04H,
    "source_04e_sha256": EXPECTED_04E_SHA256,
    "type_bias": TYPE_BIAS,
    "temperature": FROZEN_TEMPERATURE,
    "counterfactuals": {
        "full": "SAR + optical + P(clear)",
        "no_sar": "SAR zeroed; optical + P(clear) retained",
        "no_optical": "optical zeroed; P(clear)=0; SAR retained",
    },
    "attribution_metric": "normalized Jensen-Shannon divergence / ln(2)",
    "interpretation": "model dependence / counterfactual sensitivity, not physical causality",
    "calibration_scenes": 34,
    "attribution_scenes": 34,
    "high_fragility_threshold": HIGH_FRAGILITY_THRESHOLD,
    "dominance_share_threshold": DOMINANCE_SHARE,
    "counterfactual_integrity": {
        "full": attr_result["full_metrics"],
        "no_sar": attr_result["no_sar_metrics"],
        "no_optical": attr_result["no_optical_metrics"],
    },
    "attribution_summary": summary,
    "error_detection": attr_result["error_detection"],
    "test_status": "LOCKED_NOT_EVALUATED",
}

protocol_path = OUTPUT_ROOT / "FloodShift_06_protocol_record.json"
with open(protocol_path, "w") as f:
    json.dump(protocol, f, indent=2)

config = {
    "experiment": "FloodShift-06-Counterfactual-Sensor-Attribution",
    "source_04h_sha256": SHA_04H,
    "type_bias": TYPE_BIAS,
    "temperature": FROZEN_TEMPERATURE,
    "high_fragility_threshold": HIGH_FRAGILITY_THRESHOLD,
    "dominance_share_threshold": DOMINANCE_SHARE,
    "test_status": "LOCKED_NOT_EVALUATED",
}

config_path = OUTPUT_ROOT / "FloodShift_06_attribution_config.pt"
torch.save(config, config_path)

archive_path = OUTPUT_ROOT / "FloodShift_06_ARCHIVE.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [
        protocol_path,
        config_path,
        cal_tile_path,
        attr_tile_path,
        scene_path,
        class_path,
    ]:
        if Path(p).exists():
            zf.write(p, arcname=Path(p).name)

print("=" * 112)
print("P14-C15 — EXPORT")
print("=" * 112)
print("Protocol record :", protocol_path)
print("Attribution cfg :", config_path)
print("Tile evidence   :", attr_tile_path)
print("Scene evidence  :", scene_path)
print("Class evidence  :", class_path)
print("Archive         :", archive_path)
print("Frozen geo test : LOCKED_NOT_EVALUATED")
print()
print("NEXT STAGE: deterministic GeoEngine / grounded evidence packet.")
print("P14-C15 COMPLETE ✅")


## After this notebook

Send me the executed notebook or the outputs from:

- **P14-C10** — integrity check
- **P14-C11** — fragility threshold / dominance counts
- **P14-C12** — attribution summary + error-detection diagnostics
- **P14-C13** — top fragile scenes
- **P14-C14** — class-specific attribution table
- **P14-C15** — exported artifacts

Also download:

```text
FloodShift_06_ARCHIVE.zip
```

The key outputs for the next stage are:

```text
FloodShift_06_scene_evidence.csv
FloodShift_06_class_attribution.csv
FloodShift_06_attribution_config.pt
```

These become part of the deterministic evidence packet that the later VLM/LLM is allowed to consume.
